In [ ]:
import math
import pandas as pd

# FLOP grid: 1.25e16 * 2^i for i = 0..11
flop_grid = [1.25e16 * (2 ** i) for i in range(12)]

# Candidate model sizes from the paper-style family
# Units: parameters
paper_candidates = [
    75e6,    # 75M
    250e6,   # 250M
    500e6,   # 500M
    1e9,     # 1B
    2.5e9,   # 2.5B
    5e9,     # 5B
    10e9,    # 10B
    16e9,    # 16B
]

def human_count(x: float) -> str:
    if x >= 1e9:
        return f"{x/1e9:.3g}B"
    if x >= 1e6:
        return f"{x/1e6:.3g}M"
    if x >= 1e3:
        return f"{x/1e3:.3g}K"
    return f"{x:.0f}"

rows = []

for i, C in enumerate(flop_grid):
    # From:
    # C ~= 6ND
    # M = D/N in [10, 25]
    #
    # Since D = C / (6N),
    # M = D/N = C / (6N^2)
    #
    # Solving for N:
    # sqrt(C / (6 * 25)) <= N <= sqrt(C / (6 * 10))
    N_low = math.sqrt(C / 150.0)   # M = 25
    N_high = math.sqrt(C / 60.0)   # M = 10
    N_mid = math.sqrt(C / 120.0)   # M = 20

    D_mid = 20.0 * N_mid           # dataset size for M = 20

    valid_candidates = [
        human_count(n) for n in paper_candidates
        if N_low <= n <= N_high
    ]

    rows.append({
        "i": i,
        "FLOPs": C,
        "FLOPs_human": f"{C:.2e}",
        "N_low": N_low,
        "N_low_human": human_count(N_low),
        "N_mid_M20": N_mid,
        "N_mid_M20_human": human_count(N_mid),
        "N_high": N_high,
        "N_high_human": human_count(N_high),
        "D_mid_M20": D_mid,
        "D_mid_M20_human": human_count(D_mid),
        "valid_paper_candidates": ", ".join(valid_candidates) if valid_candidates else "none",
    })

df = pd.DataFrame(rows)

print("\n=== FLOP grid + valid model-size range ===")
print(df[[
    "i",
    "FLOPs_human",
    "N_low_human",
    "N_mid_M20_human",
    "N_high_human",
    "D_mid_M20_human",
    "valid_paper_candidates",
]].to_string(index=False))

# Optional: save to CSV
df.to_csv("scaling_candidates.csv", index=False)
print("\nSaved full table to scaling_candidates.csv")